In [1]:
!pip install pandas
!pip install torch
!pip install transformers
!pip install scikit-learn
!pip install numpy
!pip install urllib3

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [24]:
import torch
import numpy as np
import pandas as pd
import urllib.request
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup


# Load your dataset by simply uncomment any one of below
url = "<dataset-url>"  # dataset CSV location removed; set before running

#download and read the contents
headers = {'User-Agent': 'Mozilla/5.0'}
request = urllib.request.Request(url, headers=headers)
df = pd.read_csv(urllib.request.urlopen(request), encoding='utf-16')
# Initialize the tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')
model = BertForSequenceClassification.from_pretrained('bert-base-chinese', num_labels=4)
model.pooler = torch.nn.Linear(model.config.hidden_size, 4)  # 4 classes for multi classs classification

# Tokenize the data
tokenized = df['hex'].apply((lambda x: tokenizer.encode(x, add_special_tokens=True, max_length=512, truncation=True)))

# Pad the data
max_len = 0
for i in tokenized.values:
    if len(i) > max_len:
        max_len = len(i)
padded = np.array([i + [0]*(max_len-len(i)) for i in tokenized.values])
padded.shape

# Create attention masks
attention_mask = np.where(padded != 0, 1, 0)
attention_mask.shape

# Convert to tensors
input_ids = torch.tensor(padded)
attention_mask = torch.tensor(attention_mask)
labels = torch.tensor(df['label'])

# Create the dataset
dataset = TensorDataset(input_ids, attention_mask, labels)

# Split the dataset
train_size = int(0.4 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create the dataloaders
batch_size = 16
train_dataloader = DataLoader(train_dataset, sampler=None, batch_size=batch_size)
val_dataloader = DataLoader(val_dataset, sampler=None, batch_size=batch_size)

# Set the optimizer
optimizer = AdamW(model.parameters(), lr=5e-5, eps=1e-8)

# Set the number of epochs
epochs = 50
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

# Set the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Define the training loop
def train():
    model.train()
    total_train_loss = 0
    for step, batch in enumerate(train_dataloader):
        batch_input_ids = batch[0].to(device)
        batch_attention_mask = batch[1].to(device)
        batch_labels = batch[2].to(device)
        model.zero_grad()
        outputs = model(batch_input_ids, token_type_ids=None, attention_mask=batch_attention_mask, labels=batch_labels)
        loss = outputs[0]
        total_train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
    avg_train_loss = total_train_loss / len(train_dataloader)
    return avg_train_loss


# Train the model
for epoch in range(epochs):
    train_loss = train()
    model.eval()
    total_val_loss = 0
    predictions, true_labels = [], []
    for batch in val_dataloader:
        batch_input_ids = batch[0].to(device)
        batch_attention_mask = batch[1].to(device)
        batch_labels = batch[2].to(device)
        with torch.no_grad():
            outputs = model(batch_input_ids, token_type_ids=None, attention_mask=batch_attention_mask)
        logits = outputs[0]
        logits = logits.detach().cpu().numpy()
        label_ids = batch_labels.to('cpu').numpy()
        predictions.append(logits)
        true_labels.append(label_ids)

    flat_predictions = np.concatenate(predictions, axis=0)
    flat_predictions = np.argmax(flat_predictions, axis=1).flatten()
    flat_true_labels = np.concatenate(true_labels, axis=0)
    val_accuracy = accuracy_score(flat_true_labels, flat_predictions)
    print('Epoch:', epoch+1)
    print('Training loss:', train_loss)
    #print('Validation accuracy:', val_accuracy)

# Define the prediction function
def predict(text):
    tokenized = tokenizer.encode(text, add_special_tokens=True, max_length=max_len, truncation=True)
    padded = np.array([tokenized + [0]*(max_len-len(tokenized))])
    attention_mask = np.where(padded != 0, 1, 0)
    input_ids = torch.tensor(padded)
    attention_mask = torch.tensor(attention_mask)
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    with torch.no_grad():
        outputs = model(input_ids, token_type_ids=None, attention_mask=attention_mask)
    logits = outputs[0]
    logits = logits.detach().cpu().numpy()
    print(str(logits))
    pred = np.argmax(logits, axis=1)
    print(str(pred))

# Test the model, 01223
predict("㔷㕢愴㜶㙥㈷㑤攰㘹ㄷ㤱摥㑡㑢つ㉥摣㠲㥢㝤㉤搶捦摦㘸ㄷ摤挸ㄶ㈴㉡搹㠷㘴改搱㘷昱换㥣㌹っ㥥て昱收挷㤹扦ㅡ扣〲挹ㅢ㐱㍡㘴〶挵挱㔶ㄵ晦ㄸ㉡㜰扤慢搳㜴㔱㠳㜸捡㉢㌵昲愰捤㍤挹㝣㌵〲愵㐲戰㜹㙣㤲ㄱ改愶㑦㤱摦㠴㤴㙤ㄵ挰㜴㐱昷慢㡢戱㝥捤㔱扣〸捥搹㑦㠶攲敡㥥㘴敢〳㤰〳ㄶ㉣㜷戵㥤ㄵ㑣㙦㌴㔰㙦㜴戱㌸㔱㐵㑥ㅥ戵㤳㘳摥昹㉡㐱㜰挱㐸㤴㈷ㅣづ㙡〵慣㥡㑦ちㄷ㡤ㅢ㈲㈷慡㉣㐶㈰㔸ㄷ㑤戲㜱〲捥愴挴㌹㕥ㄲ慡㐴晡㠳㍣㜳㠱慦扤㐳㘱㐸〰〵扢戵戱㘱攷收㉤㠶挶㡤晤㑤㕣戴昸ㄳㅤ㥦㔳搲㠴〱㜹㙢㙦㌹㥢㤹て㠲㤸㕡㕦㥡㠳昶ㅣ〰〷昰戸㐷㌰㉤〰㐳晥晥〰晥晥晥晥晥晥晥晥晥晥晥晥〰〰晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥〰㠲晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥〰晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥〰〰〰〰晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥〰〰晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥〰晥〰〰〰〰晥〰〰晥〰晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥〰〰〰晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥〰晥晥〰〰晥〰〰〰〰晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥〰〰〰〰〰晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥晥")
predict("㈸扣㔱㕥〳㈱昹㙢搰敤㘴㔹搵㤱㔰昷㌲㐱愱㥡愱㔹㕣挵ㄵㅤ㘰㑤㡢扦㌱愷㥦攵晢挰㍢愲〲散〶愸㌳晤〵㝥㉦ㄶ〷ㅡ㝣㘲㝡㐷搳㘱㠶㌷㤸愵㈴㙡㕣㠳ㅥ㉥昷戹㐲㤹扥ㅡㄶ愱攷ㄸ㌹㝦戵愷愱戳㤹㜴㌳㐹㝥㜶戰〹㈰㈴慢㥢昴㤳摥㘸㜳搷户扦㈱捡挶㜵㠱晣㉥㔶㈲㘵㈲㜹㝥㘵㈳挲㙡〵㘳扦㤱㘹戵捥㜹㜰捡攰㜹㕥戳愱昸㘱ㄵ㤶扦㔲〴㐵㐵搱㜷敤〵〱㌳昷愸㉥愷㠹挸㠷搷㄰㜲戹㘷㕡㕦㈲㠳㑤㠵搳扥捣㡥㈳㈶㑢攸搶㉣挴挰摤て攳㘲㘳扦挳㈲㙣㐰㐲捡つ昶改搱㠲㙤敦ㅢ㡣扥㤶㝦㈲扥㐹㠳昳搳挴〰慣㝣㍡づ攸て㠲㜷㌰挷搲挰㤰㜲ㄲ愰㜸搶摣㉦扥㤳挴㤴㡤ㄶ㐹搱㙣敡㔷㔳㑦挰ㅦ扥㈵㉣㜸挳㔲㡦挰㑥挶〳㉢戱ㄹ昱㐹挱㌷戲㤰扥慥㍦㌳㜴ㅤ㘲挰㡣愶㌲㜴ㅦ㝦〷㥤慡攸㜰搰㤳㠹戰㍡㙤敥搳慣㝥㘸㉡㥦敦㍢㌲敡㉤㐵ㄴㄹ愳散㡤敥㤹㌱㐰慣扦㍥㕣㈰㘶搲㌱ㄱ〹㘰㤰挱挰搵搲㕤扦㙣攳㘶〳ㅥ晡ㄸ㘴戶㐰敦搲扦攵㠵㘷挴敦㑡㈵㍢搶㉦㜶㉦戸敥戲ㄳ㜹昰戱ㅥ㤰扡㝦㔹㉤挸敡敤㠵晡慦昴挷㈳㌲㤴扤挹捥愷戲ㄵ戲㥤挹摦㔱搷㔱ㄴ㉦㐶㙣昱戲〶㐳扦挱㠰㈳晥攴㝣摣つ攱㜶ㅦ晦敡摡〳㐰扦㤰〷㝥挶㤷㌳㤵㈴㈳㙣愲搴慢㠲㠰㌱㈹㕥昶㈱㜷慢㤱摦晤㙢挹㜶㍡敥㠴㐹㉣愱ㄷ搰ㄶ户㕥昴㠸㐷㔸昲㌴㙤㠵㄰㥦㐱〹㌹㐳㡦〵〵㠷攰㤶㤲ㅢ㠱昵攰㉣㍤㘹〶捤攳㍡捤挵ㄳ㌹昹ㄲ㑣挶戸㝥㑢㘱挴ち愲㡤收戶㉦㕣㠴㠷")
predict("慥愳㡣㜱〱㐲ち昴㜳ㅢ晢㐰㉤晥㜵慣捦㌱㔷㤳㘶㜴㘷㘵扣攷ぢ㉡㕥挶㜳㘱㕢扥愴ㅤ㘵㥣搴㑥㠶㔰〴㘲晤挲㥥㍣晥摣㕣愳㑣㡣晤㠷㘴搲㠸㤵晦ㅢ㝣ㅡ攰ㄶ㘰捡愴㤵㙢㑥㍦㉤昸㝣慢㝦㜵慣挹昵戵㕤㐷昰敤㠱㠰㑤㙡㝦ㄲㅡ〰搳昷㤶搲㘰愱挳㤱㘳㜸ㄶ㐹摢昹ㄸ㝥㙣摤㡦敥㉤㔱慣㍦摢㑢搷㔵㍥㜰づ挹愱㘱扤ㅡ敦㤴㑦㉦㝣㐰愱㔲晢て㐶ぢ戴愵㡣愸㝤㡦愶㄰捦㜰㥡ㄸ捡ㄲ搵挷敦㈷㡦户㈸㌷慡㝡㤴㝡扣㠱㘱㘰㙣て改戳㕡㝦㐵扣㍤㉣搰〶扤㉢㘴ㄲ㌵㥤㠶㕢攴㐲㉤㌹㜱㕦ㄳ〷晦㙦搶㘵㙣〴搶㤹挷ㅥ〴愲㔴昳愹㥡捦慥㜷捥戵㌶愹㘶㙤㍡ㄸ愲㘹㍣戰扣㜱㜱㥡㥦㥥戱㜸晥㔲㤵㉥扦〶㈹㔸㘶㠷㌲㑣攷昱㡣搳つ㔹摢扤㌵㜱昸㜷㥤搰っ昰㔶〴晢㍦昰攳愵攰摣㝢㡤㡦㘰㤶愴捥㤸㜴㑣扢㜶慦㈸挰慦㘸㐸㙢㈵慦㘳戲攰㈸挳摢〴㥥㔳〱㌱㠷愲㘹㔰ㄷ㜸ㅥ㥥扦㘶敡愵愸昲㕡㐹敥㍦扦搹㘵㔷㠶㑣㠴晦㠷晡㜰㑡晥㝦〴户㔹㥢摡㐶戹ぢ㝣扡〶㕦ㄵ㌵攷ㅦ㜰㐹昷敢㉦㘵㑦搴挵㤶㤴戵㐶ㅦ㔶㍡㥡搷㈱㜷戵㑡搶戲愶㤳〳㤶㘰㙢戸愷㉡㝥ㄷ㈴ㅣ收㈸㠷ㅡ挵㍢扣㙦攰挳晣㠳攲ㄳ㈲捤㤷㥦㜹㑤㡢〸㔸㕢㍣㘰㜸〶ㄸ㌵㌱攲㥡戴㔳攳㔲捡昴挷晦戶㍣㈱散摡ㄴ㤶㝥摣攳㔱㠴〷㐰昲㘵晤戵晤挳慡攲㠸㌹㥡づ挸㔹㑦敦摤㉦捡㙦㠹愳づ攰㍢㜷㜸㘶戱捣戴㤴㜰㌰㠱昴戶㤳搷㡣㜲〳㔲戶戸愱㔴〲慡愶攳㉣摢敢㜴昷㍡晢挲㝥昰戶㈰")
predict("晦㌸〹昰㘱捣㐰捣晦〰㠰晦㘴捥㡥戸㤵戸㑦㐱っ搸㌸〵晦っ㄰戸㌵㕥㈸㠵愶〵慡〴敦戸摤慣挵㔷㘵㡦㥢㠷戸〰㤳晦慦㌳〰慦攷㑤攴〱㔷㐱㝢戸㠰戸〰〰㈹〰〴晦㤸㔷愵㤶愹戸㐲〰㐲〷戸戴搸㐰捥㤰昵晦戸㌳㤳㠲㌸〵〱㐸㠴㡦戸㐰散扥㌵㘴愶〰㡦挰㥦㄰㌵愶㑦㐲昷挷㐴〵慤晦戸愶晦㠵戱㠲㔷㉤挲㤸搸挰㡥〰㠶搴㕦っㅣ㐵㠸〰㔴晦㥥㜵〰㙦㝤晦㔲晦ㄵ㐱〰〵㕣㠴㠰㙦㔱㄰㠵昷〰㐷っ㜵晦晦〰㜵捥㘵晦㝢㌸㐲晦昰愶㈳てㄵ搸戸〰ぢ攰っ㌰㐰捤㐴攱挵づㄶㄶ㘴挴㤴㔴㤴㤴〰㐵搴昴㔴ㄴ㌵㤴㐵昴㔷㐵㔴㌵㐵㐴㐷㐶っㄶ〰㔱㔷㡣㈷收搶〰㌷〴〴戸戸愳㙢〴〸㜳㠳㑤㐳攳㉥昹ㄷ㄰愱ㄳ㘳㍤㝥慥㕦㜴晡㔵戶㉣〰ㅥ㤰〰〰〰㌲㈴㠰愷㉦㜲㘱昸㤱昱ち㘸晥换扥㡡っ㑤㙤㕢㌴ㄱ㤲て攸ㅢ〰〰㡢扦㘵㥢㍤捥㈴戳㜰〰〰〰〰㔵〳摤〳摤〰㠴㠱㠷搷㐹搸昶〰㠶搲挴㔶搲㘶〷收ㄸㄸ捦挷户㠰〰㑢慥〰つ挷〰摡㜴〰捦戲〲昶㠴㠳〲〰〰〰〰愱〰〰㑣㑤〰〰㠷㈰〰〰挶㄰戰昱攰㔹㝡㡢晢㥣搶搶搶搶搶〱搱〲昲㌳㐳㤳戳ㄴ挴戳㐶搶搷㠶㠶攱㐷㠶㠰㌱搶㌱㌱㠰㠰搶㤹㐳晣晣㕤㌱慤扤搶捤㉥㡥搶㥦㘰搶ㄱ㠱㠱㠱㌳愱扤搶晢㐰〱攱昱搶挲搶搶〴搶〵搶搶㜶愶㡥挶搶㌱搶摡摡捡摡晢㌱㌱搶搶㌱搶搶扤搶慡㠶昵昴昵昴昴昴昴昴昴昴昴昴昴昴昴户户〰搴搴〰挴〵㤴ㄴ㔴〰搲搰〰ㄶ㠶〲收㈶收㤷ㄶ〰")
predict("挵㠱㍡㜲㠷㕦敦㘱㘵㠹昱ㄷ㑡㤸㌹晣㠹晢搶捥愹攸㜰昰扦㔱〸㠴㐳戴昵敦㙦㈰㑤搵搲戸戵㥡ㅥ㤷〶慦㜹ㄱ戸晥〶㤳㝣㜰㍣〴㤰㔷㔳㡣ㄲ攰㕡㜲㘸㠷㔳㤵摦㌱㔶摤戸㑥挱戸扥㉡つ戵㙡㕤捣挶〵〱戸㑡㜶搱㈳〷ㅥ㠷〴戰攵㕡㌳摤㍥挱㘷㍤㠳昳ㄸ㕢㜳戰㠷攵㥢㉣晡㈱捡㜸昶㘲挱户慡㤰㙤敥㙢㑡㤳㠰昲㉦搹㘴敥㜲愳ㅤづ㘵ㄷ摥昸㡢㍥慢㈱㝥〱㌲愴搹㐴㐷戸㔴昶搶搰换㌳ㄵ㍡㍡攴㍥㍢愰戱摤㌹㤳挴户搳戶㥤愲搱㕤㑥摤㠴㐳㙤㈸戵昴㘲㈸㙤㕢ㄸ㝦㈰慡㉥愴〰昲愴晣敡㙥搷晤㝢㐵㍢㔰〴㈴搶㙥㝦㡣挴㙤㡦㙥㠴改扣戶昱㘴晡㜰户㜸捤㤵㝡㙤㔰㈴捡㤶㜲㑡㘶㕦〸ㄷ㐳㉦づ㌵㥥户㥣㕢㑡扦㙥㌵晤㔹㤰搴㐷晦つ慥㙥㠷㐰㤶㜰㘶ㅢ㤸㙢㠱〳㘳㜶㕤搹〴㥣摦㝣攳㑣愸㍣㕢ㄹ捥挸㐶㑤摦㥣晢㡥挰挱攸改㙤㈴㜰㔰昴敥㥢㝥昰㜶㐰愲昹㤰戹㈳㤵㝢愱ㄷ㐵㘶扢㘶摡㤲㈴慥改㝥攵搲㔴㍣扥搲扢㉣㌱挵㡢挹㡢昹扢敥㥢挰㙣搹攳㔵㌳㈵㙤搹㈸㄰慦慣㘶挲㤶搳㜰㠶㡣つ攳㉢扣㕦㡤挵㜳搷搲晡戵慣㠳散㝦ㄱ㍥㘸㥦戰㘳㡦㤵〲㜳㠵㤳攴㘰搰㜱㝤㤷㐷㈶晦ㄹ敡捦㉢昴敥搳ㄹ㝢㠷挹㐶ぢ㉣攴㍢ㄳ愵捤换㉢攰㠷㝦晣㌶㜵搰戵搸敡㄰㘸㑤㍣ㅥ搹ち〶昰捦捣愰攱换㘱摤㉥㍡扥㈶摢㉢扥〱㕥〹㝥慦〷晤慡捦㑦㌶挶愵愶㡣愶ㅢ慦ㄴ昰㌲㤳慤摤㔵㑥愴㍣愱㠵㠷㐴ㄵ摢㠷摥搹㔵㔷昳搹捦㍦て㍢㠰ㄱ攸㝢愷㡦㠰㙣㜹㈲")

# Save the model in pt format
with open('/content/drive/My Drive/classifibert.pt', 'wb') as f:
  torch.save(model, f)




Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch: 1
Training loss: 1.6634063720703125
Epoch: 2
Training loss: 1.3148571252822876
Epoch: 3
Training loss: 1.1163132190704346
Epoch: 4
Training loss: 1.0673589706420898
Epoch: 5
Training loss: 0.9666876792907715
Epoch: 6
Training loss: 0.9279811978340149
Epoch: 7
Training loss: 1.0023956298828125
Epoch: 8
Training loss: 0.7271212935447693
Epoch: 9
Training loss: 0.8083495497703552
Epoch: 10
Training loss: 0.6222105622291565
Epoch: 11
Training loss: 0.483770489692688
Epoch: 12
Training loss: 0.5092432498931885
Epoch: 13
Training loss: 0.5891497135162354
Epoch: 14
Training loss: 0.4578172266483307
Epoch: 15
Training loss: 0.41195639967918396
Epoch: 16
Training loss: 0.414573609828949
Epoch: 17
Training loss: 0.34569045901298523
Epoch: 18
Training loss: 0.2682941257953644
Epoch: 19
Training loss: 0.26935014128685
Epoch: 20
Training loss: 0.2130683809518814
Epoch: 21
Training loss: 0.2231561839580536
Epoch: 22
Training loss: 0.18686871230602264
Epoch: 23
Training loss: 0.160416409373283